## *preparação de dados através de geração de dados com LLMs*

In [ ]:
import os
import time
import random
import re
import requests
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from text_features import truncate_text

# ── carregar .env ──────────────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    from pathlib import Path
    env_path = Path("../.env")
    load_dotenv(dotenv_path=env_path)
    print(f".env carregado de: {env_path.resolve()}")
except ImportError:
    print("AVISO: python-dotenv não instalado. Instalar com: pip install python-dotenv")

try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    print("AVISO: groq não instalado. Instalar com: pip install groq")
    GROQ_AVAILABLE = False

# ── configuração ──────────────────────────────────────────────────────────────
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if GROQ_API_KEY:
    print(f"GROQ_API_KEY carregada (termina em ...{GROQ_API_KEY[-6:]})")
else:
    print("AVISO: GROQ_API_KEY não encontrada. Preencher ../.env")

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    print(f"HF_TOKEN carregado (termina em ...{HF_TOKEN[-6:]})")
else:
    print("AVISO: HF_TOKEN não encontrado – necessário para descarregar modelos Gemma (modelo gated)")

OUTPUT_DIR = "../datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N_GROQ_PER_CLASS = 150    # textos a gerar via Groq por classe
N_HF_PER_CLASS   = 150    # textos a gerar via transformers por classe
MAX_PER_CLASS    = 1000   # máximo por classe no dataset final

print("Setup concluído.")

## *tópicos e prompts*


In [ ]:
SCIENCE_TOPICS = [
    # química
    "covalent bonds in chemistry",
    "ionic bonds and electrostatic attraction",
    "polyphenols and their antioxidant properties",
    "anthocyanins as plant pigments and their health benefits",
    "PFAS per- and polyfluoroalkyl substances as forever chemicals",
    "diamond formation under extreme pressure in Earth's mantle",
    "X-ray diffraction and crystal structure analysis",
    "enzyme catalysis and substrate binding",
    "Michaelis-Menten enzyme kinetics and the KM constant",
    "steady-state conditions in biochemical reactions",
    # biologia / evolução
    "Lamarck's theory of evolution and inheritance of acquired traits",
    "Darwin's theory of natural selection",
    "Darwin and the Galápagos Islands observations",
    "the origin of the genetic code and RNA world hypothesis",
    "codon usage bias in genetics",
    "epigenetics and heritable changes in gene expression",
    "epigenetic adaptation of giant pandas to bamboo diet",
    "abiogenesis and the chemical origin of life",
    "zebrafish as a model organism in biomedical research",
    "emergent properties in complex biological systems",
    "the origin of angiosperms and flowering plants",
    "the RNA world and early self-replicating molecules",
    "photosynthesis and the Great Oxygenation Event",
    # biomedicina / saúde
    "H5N1 avian influenza virus and pandemic risk",
    "the main causes and risk factors of lung cancer",
    "Alzheimer's disease current treatments and cholinesterase inhibitors",
    "rivastigmine in the treatment of Alzheimer's disease",
    "Type I and Type II statistical errors in hypothesis testing",
    # física / astronomia
    "Schrödinger's cat thought experiment and quantum superposition",
    "quantum entanglement and correlated particles",
    "black hole formation from massive star collapse",
    "supermassive black holes at galactic centers",
    "the future of the Solar System when the Sun becomes a red giant",
    # geociências / meteorologia
    "plate tectonics and the movement of Earth's lithosphere",
    "anticyclones and high-pressure weather systems",
    "subduction zones and convergent plate boundaries",
]

# três variações de prompt para cada tópico → gera diversidade estilística
PROMPT_TEMPLATES = [
    "Write a concise scientific paragraph of approximately 100 words explaining {topic}. Be factual and educational.",
    "In about 100 words, describe {topic}. Use clear, scientific language suitable for a general educated audience.",
    "Provide a brief scientific overview of {topic} in approximately 100 words. Include key concepts and their significance.",
]

def build_prompts(topics, templates):
    pairs = []
    for topic in topics:
        for tmpl in templates:
            pairs.append((topic,tmpl.format(topic=topic)))
    random.shuffle(pairs)
    return pairs

prompts = build_prompts(SCIENCE_TOPICS, PROMPT_TEMPLATES)
print(f"total de prompts disponíveis: {len(prompts)}")
print(f"exemplo:\n  {prompts[0][1]}")

## *geração via Groq API*

In [ ]:
GROQ_MODELS = {
    "meta":   "llama-3.3-70b-versatile",
    "openai": "openai/gpt-oss-120b",
}

def clean_generated_text(text):
    """remove prefixos comuns que os modelos às vezes adicionam."""
    text = text.strip()
    prefixes = [
        r"^here'?s? (?:a |is )?(brief |short |concise )?(?:overview|explanation|paragraph|description)[^:]*:\s*",
        r"^sure[!,.]?\s*",
        r"^of course[!,.]?\s*",
        r"^certainly[!,.]?\s*",
    ]
    for pat in prefixes:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)
    return text.strip()


def groq_generate(client, model_id, prompt, max_tokens=250, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", 
                           "content": prompt}],
                max_tokens=max_tokens,
                temperature=0.7,
            )
            return clean_generated_text(response.choices[0].message.content)
        except Exception as e:
            print(f"  ERROR (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(10 * (attempt + 1))
    return None


def generate_class_texts(client, label, model_id, prompts, n_samples, sleep_sec=1.5):
    rows = []
    prompts_to_use = prompts[:n_samples]
    for i, (topic, prompt) in enumerate(prompts_to_use):
        text = groq_generate(client, model_id, prompt)
        if text:
            rows.append({"text": text, "label": label})
        if (i + 1) % 10 == 0:
            print(f"  [{label}] {i+1}/{len(prompts_to_use)} textos gerados...")
        time.sleep(sleep_sec)
    print(f"  [{label}] concluído: {len(rows)} textos válidos.")
    return rows


print("funções Groq definidas.")

In [ ]:
checkpoint_path = f"{OUTPUT_DIR}/groq_generated.csv"

if os.path.exists(checkpoint_path):
    df_groq = pd.read_csv(checkpoint_path)
    print(f"Checkpoint encontrado: a carregar de {checkpoint_path}...")
    print(df_groq['label'].value_counts())
elif GROQ_AVAILABLE and GROQ_API_KEY:
    client = Groq(api_key=GROQ_API_KEY)
    groq_rows = []

    for label, model_id in GROQ_MODELS.items():
        print(f"\n-> A gerar textos para classe '{label}' com modelo '{model_id}'...")
        shuffled = prompts.copy()
        random.shuffle(shuffled)
        rows = generate_class_texts(
            client, label, model_id, shuffled,
            n_samples=N_GROQ_PER_CLASS,
            sleep_sec=1.5
        )
        groq_rows.extend(rows)

    df_groq = pd.DataFrame(groq_rows)
    print(f"\ntotal gerado via Groq: {len(df_groq)}")
    print(df_groq['label'].value_counts())
    df_groq.to_csv(checkpoint_path, index=False)
    print(f"checkpoint guardado em {checkpoint_path}")
else:
    df_groq = pd.DataFrame(columns=['text', 'label'])
    print("Groq API nao disponivel e sem checkpoint, df_groq vazio.")

## 3. *geração via Ollama (gemma)*
requer o [Ollama](https://ollama.com) instalado e a correr localmente:
```bash
ollama serve          # iniciar o servidor
ollama pull gemma3    # descarregar o modelo (primeira vez)
```

In [ ]:
OLLAMA_MODEL = "gemma3:1b"
OLLAMA_URL   = "http://localhost:11434/api/generate"
checkpoint_path_google = f"{OUTPUT_DIR}/ollama_generated.csv"

df_google = pd.DataFrame(columns=['text', 'label'])

if os.path.exists(checkpoint_path_google):
    df_google = pd.read_csv(checkpoint_path_google)
    print(f"Checkpoint encontrado: a carregar de {checkpoint_path_google}...")
    print(df_google['label'].value_counts())
else:
    def ollama_generate(prompt, model=OLLAMA_MODEL, max_tokens=250, retries=3):
        for attempt in range(retries):
            try:
                resp = requests.post(OLLAMA_URL, json={
                    "model": model,
                    "prompt": prompt,
                    "stream": False,
                    "options": {"num_predict": max_tokens, 
                                "temperature": 0.7},
                }, timeout=120)
                resp.raise_for_status()
                return clean_generated_text(resp.json()["response"])
            except Exception as e:
                print(f"  ERROR (tentativa {attempt+1}/{retries}): {e}")
                if attempt < retries - 1:
                    time.sleep(5 * (attempt + 1))
        return None

    try:
        ping = requests.get("http://localhost:11434/api/tags",timeout=5)
        available = [m["name"] for m in ping.json().get("models", [])]
        print("modelos Ollama disponíveis:", available)
        if not any(OLLAMA_MODEL in m for m in available):
            print(f"AVISO: modelo '{OLLAMA_MODEL}' não encontrado! Correr: ollama pull {OLLAMA_MODEL}")

        print(f"\n-> A gerar textos 'google' com Ollama ({OLLAMA_MODEL})...")
        google_rows = []
        shuffled_hf = prompts.copy()
        random.shuffle(shuffled_hf)
        for i, (topic, prompt) in enumerate(shuffled_hf[:N_HF_PER_CLASS]):
            text = ollama_generate(prompt)
            if text:
                google_rows.append({"text": text, "label": "google"})
            if (i + 1) % 10 == 0:
                print(f"  [google] {i+1}/{N_HF_PER_CLASS} textos gerados...")

        df_hf_google = pd.DataFrame(google_rows)
        print(f"  [google] concluído: {len(df_hf_google)} textos válidos.")
        df_hf_google.to_csv(checkpoint_path_google, index=False)
        print(f"Checkpoint guardado em {checkpoint_path_google}.")

    except requests.exceptions.ConnectionError:
        print("Ollama não está a correr! Iniciar com: ollama serve")
        print("Sem checkpoint e sem Ollama: df_hf_google vazio.")

## *dados Anthropic*

textos gerados diretamente pelo Claude (Haiku 4.6) sobre tópicos científicos.  

In [ ]:
ANTHROPIC_CSV = f"{OUTPUT_DIR}/claude_generated.csv"

try:
    df_anthropic = pd.read_csv(ANTHROPIC_CSV)
    df_anthropic = df_anthropic.dropna(subset=['text'])
    df_anthropic = df_anthropic[df_anthropic['text'].apply(lambda x: isinstance(x, str) and len(x.split()) >= 60)]
    df_anthropic['label'] = 'anthropic'
    print(f"Anthropic: {len(df_anthropic)} textos")
except Exception as e:
    print(f"Falhou a carregar {ANTHROPIC_CSV}: {e}")
    df_anthropic = pd.DataFrame(columns=['text', 'label'])


## *dados Human via Wikipedia*

parágrafos de artigos científicos da *Wikipedia*.

In [ ]:
WIKI_API        = 'https://en.wikipedia.org/w/api.php'
WIKI_CHECKPOINT = f'{OUTPUT_DIR}/wikipedia_human.csv'
WIKI_HEADERS    = {'User-Agent': 'PrepareDataBot/1.0 (academic)'}

if os.path.exists(WIKI_CHECKPOINT):
    df_human = pd.read_csv(WIKI_CHECKPOINT)
    print(f'Checkpoint encontrado: {len(df_human)} textos human')
else:
    print('Checkpoint não encontrado: a gerar dados Wikipedia...')

    def search_wiki_titles(query, limit=2):
        params = {
            'action': 'query',
            'list': 'search',
            'srsearch': query,
            'srlimit': limit,
            'format': 'json',
        }
        try:
            resp = requests.get(WIKI_API, params=params, headers=WIKI_HEADERS, timeout=15)
            resp.raise_for_status()
            return [r['title'] for r in resp.json()['query']['search']]
        except Exception as e:
            print(f"  erro na pesquisa '{query}': {e}")
            return []

    def get_wiki_paragraphs(title, min_words=80):
        params = {
            'action': 'query',
            'prop': 'extracts',
            'explaintext': True,
            'exsectionformat': 'plain',
            'titles': title,
            'format': 'json',
            'redirects': 1,
        }
        try:
            resp = requests.get(WIKI_API, params=params, headers=WIKI_HEADERS, timeout=15)
            resp.raise_for_status()
            pages = resp.json()['query']['pages']
            page = next(iter(pages.values()))
            raw = page.get('extract', '')
            paragraphs = [p.strip() for p in raw.split('\n') if p.strip()]
            return [' '.join(p.split()) for p in paragraphs if len(p.split()) >= min_words]
        except Exception as e:
            print(f"  erro ao obter '{title}': {e}")
            return []

    human_rows = []
    seen_titles = set()

    for i, topic in enumerate(SCIENCE_TOPICS):
        titles = search_wiki_titles(topic, limit=2)
        for title in titles:
            if title in seen_titles:
                continue
            seen_titles.add(title)
            paragraphs = get_wiki_paragraphs(title)
            for p in paragraphs:
                human_rows.append({'text': p, 
                                   'label': 'human'})
            time.sleep(0.5)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(SCIENCE_TOPICS)} tópicos processados: {len(human_rows)} parágrafos até agora...")

    df_human = pd.DataFrame(human_rows).drop_duplicates(subset='text').reset_index(drop=True)
    print(f'\nWikipedia: {len(df_human)} parágrafos obtidos')
    df_human.to_csv(WIKI_CHECKPOINT, index=False)
    print(f'Checkpoint guardado em {WIKI_CHECKPOINT}')


## *combinar, truncar e balancear*

In [ ]:
dfs = []

for name, df in [
    ('groq (meta + openai)', df_groq),
    ('google',            df_google),
    ('anthropic',            df_anthropic),
    ('human',    df_human),
]:
    if len(df) > 0:
        dfs.append(df[['text','label']])
        print(f"{name}: {len(df)} registos")

df_all = pd.concat(dfs, ignore_index=True)
df_all = df_all.dropna(subset=['text','label']).drop_duplicates(subset='text')
df_all['label'] = df_all['label'].str.lower().str.strip()

print(f'total antes de truncar: {df_all.shape}')
print(df_all['label'].value_counts())

In [ ]:
df_all['text'] = df_all['text'].apply(truncate_text)
df_all = df_all.dropna(subset=['text']).reset_index(drop=True)

print(f"após truncar (80-120 palavras): {df_all.shape}")
print(df_all['label'].value_counts())
print("\nestatísticas de comprimento (nr. palavras):")
print(df_all['text'].apply(lambda x: len(x.split())).describe().round(1))

In [ ]:
# ── balancear classes ─────────────────────────────────────────────────────────
TARGET_CLASSES = ['meta', 'google', 'anthropic', 'openai', 'human']

df_all = df_all[df_all['label'].isin(TARGET_CLASSES)]

def balance_class(x, max_n, min_n=50):
    if len(x) == 0:
        return x
    if len(x) >= max_n:
        return x.sample(n=max_n, random_state=SEED)
    elif len(x) >= min_n:
        return x.sample(n=max_n, replace=True, random_state=SEED)
    else:
        print(f"AVISO: classe '{x['label'].iloc[0]}' contém apenas {len(x)} amostras (mínimo: {min_n})")
        return x

counts = df_all.groupby('label').size()
print("contagens antes de balancear:", counts.to_dict())
actual_max = min(MAX_PER_CLASS, counts.max())

df_balanced = (
    df_all.groupby('label', group_keys=False)
    .apply(lambda x: balance_class(x, actual_max))
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

print(f"\nApós balancear (MAX_PER_CLASS={actual_max}):")
print(df_balanced['label'].value_counts())

## *divisão e guardar*

In [ ]:
# ── split treino / validação / teste ─────────────────────────────────────────
df_train, df_temp = train_test_split(
    df_balanced, test_size=0.3,
    stratify=df_balanced['label'], random_state=SEED
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5,
    stratify=df_temp['label'], random_state=SEED
)

print(f"treino: {len(df_train)}  validação: {len(df_val)}  teste: {len(df_test)}")
print("\ndistribuição de treino:")
print(df_train['label'].value_counts())


def save_format(df, prefix, path):
    df = df[['text','label']].reset_index(drop=True).copy()
    df.insert(0, 'ID', [f'{prefix}-{i+1}' for i in range(len(df))])
    df.columns = ['ID','Text','Label']
    df['Label'] = df['Label'].str.capitalize()
    df.to_csv(path, sep=';', index=False)
    print(f"Guardado: {path} ({len(df)} registos)")


save_format(df_train, 'TR2', f'{OUTPUT_DIR}/dataset_v2_train.csv')
save_format(df_val,   'VL2', f'{OUTPUT_DIR}/dataset_v2_val.csv')
save_format(df_test,  'TE2', f'{OUTPUT_DIR}/dataset_v2_test.csv')

save_format(df_balanced, 'ALL2', f'{OUTPUT_DIR}/dataset_v2_full.csv')